In [1]:
pip list

Package                 Version
----------------------- -----------
absl-py                 2.4.0
accelerate              1.12.0
asttokens               3.0.1
astunparse              1.6.3
certifi                 2025.11.12
charset-normalizer      3.4.4
colorama                0.4.6
comm                    0.2.3
contourpy               1.3.3
cycler                  0.12.1
debugpy                 1.8.19
decorator               5.2.1
enum34                  1.1.10
executing               2.2.1
filelock                3.20.1
flatbuffers             25.12.19
fonttools               4.62.1
fsspec                  2025.12.0
gast                    0.7.0
google-pasta            0.2.0
graphviz                0.21
grpcio                  1.80.0
h5py                    3.14.0
huggingface-hub         0.36.0
idna                    3.11
ipykernel               7.1.0
ipython                 9.8.0
ipython_pygments_lexers 1.1.1
jedi                    0.19.2
Jinja2                  3.1.6
joblib      

In [1]:
import torch

# 1. Cek apakah GPU (CUDA) tersedia
gpu_tersedia = torch.cuda.is_available()
print(f"Apakah GPU tersedia? {gpu_tersedia}")

if gpu_tersedia:
    print(f"Nama GPU: {torch.cuda.get_device_name(0)}")

# 2. Cek tensor/model Anda berada di mana
# Membuat tensor biasa (secara default masuk ke CPU)
x = torch.rand(3, 3)
print(f"Tensor x berada di: {x.device}") 

# Memindahkan tensor ke GPU (jika tersedia)
#if gpu_tersedia:
    #x = x.to('cuda')
    #print(f"Setelah dipindah, tensor x berada di: {x.device}")

Apakah GPU tersedia? False
Tensor x berada di: cpu


In [6]:
# ── REAL-TIME PREDICTION + ALERT ─────────────────────────────────────
#INI BAGUS
import winsound
import cv2
import time
from ultralytics import YOLO
from collections import deque, Counter

# ===============================
# LOAD MODEL
# ===============================
model = YOLO(
    r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\best1.pt'
)

EMOTIONS = list(model.names.values())

COLORS = {
    'Angry': (0, 0, 255),
    'Drowsy': (0, 128, 0),
    'Netral': (0, 255, 0),
    'Sad': (128, 0, 128),
}

THRESHOLD = {
    "Angry": 0.60,
    "Drowsy": 0.55,
    "Netral": 0.60,
    "Sad": 0.90   # sad dibuat lebih rendah karena subtle
}

min_conf = THRESHOLD.get("Netral", 0.50)

# ===============================
# FACE DETECTOR
# ===============================
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

cap = cv2.VideoCapture(0)

# ===============================
# FPS
# ===============================
prev_time = time.time()

# ===============================
# EMOTION BUFFER
# kecilkan buffer agar responsif
# ===============================
emotion_buffer = deque(maxlen=3)

print("Tekan 'q' untuk keluar")

while True:

    start_time = time.time()

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(60, 60)
    )

    for (x, y, w, h) in faces:

        # ===============================
        # FACE CROP
        # ===============================
        face_crop = gray[y:y+h, x:x+w]

        # CLAHE enhancement
        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8, 8)
        )

        face_crop = clahe.apply(face_crop)

        # Resize
        face_crop = cv2.resize(face_crop, (48, 48))

        # Convert ke 3-channel
        face_crop = cv2.cvtColor(
            face_crop,
            cv2.COLOR_GRAY2BGR
        )

        # ===============================
        # PREDICTION
        # ===============================
        results = model.predict(
            face_crop,
            imgsz=64,
            verbose=False
        )

        probs = results[0].probs

        top1 = probs.top1
        top1_conf = float(probs.top1conf)

        top2_conf = float(probs.top5conf[1]) if len(probs.top5conf) > 1 else 0.0
        emotion = EMOTIONS[top1]

        # ===============================
        # CONFIDENCE GAP (anti-overlap trick)
        # ===============================
        confidence_gap = top1_conf - top2_conf

        # ===============================
        # CONFIDENCE FILTER
        # ===============================
        if top1_conf < min_conf:
            continue

        # ===============================
        # UPDATE BUFFER
        # ===============================
        emotion_buffer.append((emotion, top1_conf))

        # Majority voting
        score = {}

        for e, c in emotion_buffer:
            score[e] = score.get(e, 0) + c

        stable_emotion = max(score, key=score.get)
        # ===============================
        # SPECIAL RULE FOR SAD
        # Sad sering subtle
        # ===============================
        if emotion == "Sad":
            if top1_conf > 0.30:
                stable_emotion = "Sad"
        else:
            if top1_conf < min_conf:
                continue
        
        
        # ===============================
        # DROWSY ALERT
        # ===============================
        if stable_emotion == 'Drowsy' and top1_conf > 0.70:
            winsound.Beep(1000, 300)
            
        if stable_emotion == "Sad":
            if score["Sad"] < 1.2:   # akumulasi confidence
                stable_emotion = "Netral"
        
        if emotion == "Sad" and top1_conf < 0.60 and confidence_gap < 0.20:
            continue
        # ===============================
        # COLOR
        # ===============================
        color = COLORS.get(
            stable_emotion,
            (255, 255, 255)
        )

        # ===============================
        # DRAW BBOX
        # ===============================
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            color,
            2
        )

        label = f"{stable_emotion} {top1_conf*100:.0f}%"

        cv2.rectangle(
            frame,
            (x, y-30),
            (x+w, y),
            color,
            -1
        )

        cv2.putText(
            frame,
            label,
            (x+5, y-8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255,255,255),
            2
        )

    # ===============================
    # FPS & LATENCY
    # ===============================
    end_time = time.time()

    latency = (end_time - start_time) * 1000

    fps = 1 / (end_time - prev_time)
    prev_time = end_time

    # ===============================
    # DISPLAY METRICS
    # ===============================
    cv2.putText(
        frame,
        f"FPS: {fps:.1f}",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0,255,255),
        2
    )

    cv2.putText(
        frame,
        f"Latency: {latency:.1f} ms",
        (10, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0,255,255),
        2
    )

    cv2.imshow(
        'Emotion Recognition — YOLOv8n',
        frame
    )

    # Exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Tekan 'q' untuk keluar


In [1]:
# ── REAL-TIME PREDICTION + ALERT (YOLOv8n-face) ──────────────────────
import winsound
import cv2
import time
from ultralytics import YOLO
from collections import deque

# ===============================
# LOAD MODEL
# ===============================
emotion_model = YOLO(
    r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\best1.pt'
)

face_model = YOLO(
    r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\yolov8n-face.pt'
)

EMOTIONS = list(emotion_model.names.values())

COLORS = {
    'Angry'  : (0,   0,   255),
    'Drowsy' : (0,   128, 0  ),
    'Netral' : (0,   255, 0  ),
    'Sad'    : (128, 0,   128),
}

THRESHOLD = {
    "Angry"  : 0.60,
    "Drowsy" : 0.55,
    "Netral" : 0.60,
    "Sad"    : 0.30,
}

min_conf = THRESHOLD.get("Netral", 0.50)

# ===============================
# KAMERA
# ===============================
cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)

# Flush frame awal
for _ in range(5):
    cap.read()

if not cap.isOpened():
    raise RuntimeError("[ERROR] Kamera tidak bisa dibuka.")

# ===============================
# STATE
# ===============================
prev_time    = time.time()
emotion_buffer = deque(maxlen=3)

print("Tekan 'q' untuk keluar")

while True:
    start_time = time.time()

    ret, frame = cap.read()
    if not ret or frame is None:
        continue

    frame = cv2.resize(frame, (640, 480))

    # ===============================
    # DETEKSI WAJAH — YOLOv8n-face
    # ===============================
    face_results = face_model.predict(
        frame,
        conf=0.5,
        imgsz=640,
        verbose=False
    )

    boxes = face_results[0].boxes

    if boxes is None or len(boxes) == 0:
        # Tidak ada wajah — tetap tampilkan frame
        pass
    else:
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            # Padding sedikit
            H, W = frame.shape[:2]
            PAD  = 10
            x1c  = max(0, x1 - PAD)
            y1c  = max(0, y1 - PAD)
            x2c  = min(W, x2 + PAD)
            y2c  = min(H, y2 + PAD)

            # ===============================
            # CROP + PREPROCESS
            # ===============================
            face_crop = frame[y1c:y2c, x1c:x2c]

            if face_crop.size == 0:
                continue

            # Grayscale → CLAHE → BGR (sama seperti kode lama)
            gray_crop = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
            clahe     = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            gray_crop = clahe.apply(gray_crop)
            gray_crop = cv2.resize(gray_crop, (48, 48))
            face_input = cv2.cvtColor(gray_crop, cv2.COLOR_GRAY2BGR)

            # ===============================
            # PREDIKSI EMOSI
            # ===============================
            results    = emotion_model.predict(face_input, imgsz=64, verbose=False)
            probs      = results[0].probs
            top1       = probs.top1
            top1_conf  = float(probs.top1conf)
            top2_conf  = float(probs.top5conf[1]) if len(probs.top5conf) > 1 else 0.0
            emotion    = EMOTIONS[top1]
            conf_gap   = top1_conf - top2_conf

            # ── Filter Sad yang lemah ──────────────────────────────
            if emotion == "Sad" and top1_conf < 0.60 and conf_gap < 0.20:
                continue

            # ── Filter confidence umum ────────────────────────────
            if emotion != "Sad" and top1_conf < min_conf:
                continue

            # ===============================
            # BUFFER + MAJORITY VOTING
            # ===============================
            emotion_buffer.append((emotion, top1_conf))

            score = {}
            for e, c in emotion_buffer:
                score[e] = score.get(e, 0) + c

            stable_emotion = max(score, key=score.get)

            # ── Aturan khusus Sad ──────────────────────────────────
            if emotion == "Sad" and top1_conf > 0.30:
                stable_emotion = "Sad"

            if stable_emotion == "Sad" and score.get("Sad", 0) < 1.2:
                stable_emotion = "Netral"

            # ===============================
            # DROWSY ALERT
            # ===============================
            if stable_emotion == "Drowsy" and top1_conf > 0.70:
                winsound.Beep(1000, 300)

            # ===============================
            # GAMBAR HASIL
            # ===============================
            color = COLORS.get(stable_emotion, (255, 255, 255))
            label = f"{stable_emotion} {top1_conf*100:.0f}%"

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.rectangle(frame, (x1, y1 - 30), (x2, y1), color, -1)
            cv2.putText(frame, label, (x1 + 5, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # ===============================
    # FPS & LATENCY
    # ===============================
    end_time = time.time()
    latency  = (end_time - start_time) * 1000
    fps      = 1.0 / (end_time - prev_time + 1e-9)
    prev_time = end_time

    cv2.putText(frame, f"FPS: {fps:.1f}",          (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    cv2.putText(frame, f"Latency: {latency:.1f} ms", (10, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    cv2.imshow('Emotion Recognition — YOLOv8n-face', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("[INFO] Selesai.")

Tekan 'q' untuk keluar
[INFO] Selesai.


FOR REPORT

In [9]:
# ── REAL-TIME PREDICTION + ALERT + REPORT (YOLOv8n-face) ─────────────
import winsound
import cv2
import time
import csv
import os
from datetime import datetime
from ultralytics import YOLO
from collections import deque

# ===============================
# LOAD MODEL
# ===============================
emotion_model = YOLO(
    r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\best1.pt'
)
face_model = YOLO(
    r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\yolov8n-face.pt'
)

EMOTIONS = list(emotion_model.names.values())

COLORS = {
    'Angry'  : (0,   0,   255),
    'Drowsy' : (0,   128, 0  ),
    'Netral' : (0,   255, 0  ),
    'Sad'    : (128, 0,   128),
}

THRESHOLD = {
    "Angry"  : 0.60,
    "Drowsy" : 0.55,
    "Netral" : 0.60,
    "Sad"    : 0.30,
}

min_conf = THRESHOLD.get("Netral", 0.50)

# ===============================
# KAMERA
# ===============================
cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
for _ in range(5):
    cap.read()
if not cap.isOpened():
    raise RuntimeError("[ERROR] Kamera tidak bisa dibuka.")

# ===============================
# STATE
# ===============================
prev_time      = time.time()
emotion_buffer = deque(maxlen=3)

# ── REPORT STATE ──────────────────────────────────────────────────────
session_start      = datetime.now()
emotion_counts     = {e: 0 for e in EMOTIONS}   # total frame per emosi
emotion_durations  = {e: 0.0 for e in EMOTIONS}  # total detik per emosi
conf_accum         = {e: [] for e in EMOTIONS}    # kumpulkan conf untuk rata-rata

last_stable        = None
last_stable_start  = time.time()

report_interval    = 5.0   # detik — seberapa sering overlay di-refresh
last_report_time   = time.time()

# Baris ringkasan untuk overlay (diperbarui tiap report_interval detik)
overlay_lines      = []

# ── CSV LOG ───────────────────────────────────────────────────────────
log_dir  = r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika'
log_path = os.path.join(log_dir, f"emotion_log_{session_start.strftime('%Y%m%d_%H%M%S')}.csv")

csv_file   = open(log_path, 'w', newline='', encoding='utf-8')
csv_writer = csv.writer(csv_file)
csv_writer.writerow(['timestamp', 'emotion', 'confidence', 'fps', 'latency_ms'])

print(f"[INFO] Log disimpan ke: {log_path}")
print("Tekan 'q' untuk keluar  |  's' untuk simpan snapshot")

# ── HELPER: buat overlay report ───────────────────────────────────────
def build_overlay(emotion_durations, emotion_counts, conf_accum, session_start):
    lines = []
    elapsed = (datetime.now() - session_start).total_seconds()
    lines.append(f"Sesi : {int(elapsed//60):02d}:{int(elapsed%60):02d}")
    lines.append("─────────────────────")
    dominant = max(emotion_durations, key=emotion_durations.get)
    for e in EMOTIONS:
        dur  = emotion_durations[e]
        pct  = (dur / elapsed * 100) if elapsed > 0 else 0
        avg_c = (sum(conf_accum[e]) / len(conf_accum[e]) * 100) if conf_accum[e] else 0
        marker = " ◀" if e == dominant else ""
        lines.append(f"{e:<8} {dur:5.1f}s {pct:4.0f}% c:{avg_c:3.0f}%{marker}")
    lines.append("─────────────────────")
    lines.append(f"Dominan: {dominant}")
    return lines

# ── HELPER: gambar overlay semi-transparan ────────────────────────────
def draw_report_overlay(frame, lines, x=10, y=75):
    font       = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.48
    thickness  = 1
    pad        = 6
    line_h     = 18

    # ukuran kotak
    max_w = max(cv2.getTextSize(l, font, font_scale, thickness)[0][0] for l in lines)
    box_h = len(lines) * line_h + pad * 2
    box_w = max_w + pad * 2

    # overlay transparan
    overlay = frame.copy()
    cv2.rectangle(overlay, (x, y), (x + box_w, y + box_h), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

    for i, line in enumerate(lines):
        ty = y + pad + (i + 1) * line_h - 3
        # warna baris emosi sesuai COLORS
        color = (200, 200, 200)
        for emo, col in COLORS.items():
            if line.strip().startswith(emo):
                color = col
                break
        if "Dominan" in line or "Sesi" in line or "─" in line:
            color = (0, 220, 220)
        cv2.putText(frame, line, (x + pad, ty), font, font_scale, color, thickness)

# ─────────────────────────────────────────────────────────────────────
while True:
    start_time = time.time()

    ret, frame = cap.read()
    if not ret or frame is None:
        continue

    frame = cv2.resize(frame, (640, 480))

    # ===============================
    # DETEKSI WAJAH — YOLOv8n-face
    # ===============================
    face_results = face_model.predict(frame, conf=0.5, imgsz=640, verbose=False)
    boxes        = face_results[0].boxes

    stable_emotion = None

    if boxes is not None and len(boxes) > 0:
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            H, W = frame.shape[:2]
            PAD  = 10
            x1c  = max(0, x1 - PAD);  y1c = max(0, y1 - PAD)
            x2c  = min(W, x2 + PAD);  y2c = min(H, y2 + PAD)

            face_crop = frame[y1c:y2c, x1c:x2c]
            if face_crop.size == 0:
                continue

            # Preprocess
            gray_crop  = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
            clahe      = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            gray_crop  = clahe.apply(gray_crop)
            gray_crop  = cv2.resize(gray_crop, (48, 48))
            face_input = cv2.cvtColor(gray_crop, cv2.COLOR_GRAY2BGR)

            # ── Prediksi emosi ────────────────────────────────────
            results   = emotion_model.predict(face_input, imgsz=64, verbose=False)
            probs     = results[0].probs
            top1      = probs.top1
            top1_conf = float(probs.top1conf)
            top2_conf = float(probs.top5conf[1]) if len(probs.top5conf) > 1 else 0.0
            emotion   = EMOTIONS[top1]
            conf_gap  = top1_conf - top2_conf

            # Filter Sad lemah
            if emotion == "Sad" and top1_conf < 0.60 and conf_gap < 0.20:
                continue
            if emotion != "Sad" and top1_conf < min_conf:
                continue

            # Buffer + majority voting
            emotion_buffer.append((emotion, top1_conf))
            score = {}
            for e, c in emotion_buffer:
                score[e] = score.get(e, 0) + c
            stable_emotion = max(score, key=score.get)

            # Aturan khusus Sad
            if emotion == "Sad" and top1_conf > 0.30:
                stable_emotion = "Sad"
            if stable_emotion == "Sad" and score.get("Sad", 0) < 1.2:
                stable_emotion = "Netral"

            # ── Drowsy alert ──────────────────────────────────────
            if stable_emotion == "Drowsy" and top1_conf > 0.70:
                winsound.Beep(1000, 300)

            # ── Gambar bbox ───────────────────────────────────────
            color = COLORS.get(stable_emotion, (255, 255, 255))
            label = f"{stable_emotion} {top1_conf*100:.0f}%"
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.rectangle(frame, (x1, y1 - 30), (x2, y1), color, -1)
            cv2.putText(frame, label, (x1 + 5, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            # ── Akumulasi statistik ───────────────────────────────
            now = time.time()
            emotion_counts[stable_emotion] += 1
            conf_accum[stable_emotion].append(top1_conf)

            # Hitung durasi emosi stabil
            if stable_emotion != last_stable:
                last_stable       = stable_emotion
                last_stable_start = now
            else:
                emotion_durations[stable_emotion] += now - last_stable_start
                last_stable_start = now

            # ── CSV log ───────────────────────────────────────────
            end_time_log = time.time()
            fps_log      = 1.0 / (end_time_log - prev_time + 1e-9)
            lat_log      = (end_time_log - start_time) * 1000
            csv_writer.writerow([
                datetime.now().strftime('%H:%M:%S.%f')[:-3],
                stable_emotion,
                f"{top1_conf:.4f}",
                f"{fps_log:.1f}",
                f"{lat_log:.1f}"
            ])

    # ===============================
    # REFRESH OVERLAY REPORT
    # ===============================
    now = time.time()
    if now - last_report_time >= report_interval:
        overlay_lines     = build_overlay(emotion_durations, emotion_counts,
                                          conf_accum, session_start)
        last_report_time  = now

    if overlay_lines:
        draw_report_overlay(frame, overlay_lines)

    # ===============================
    # FPS & LATENCY
    # ===============================
    end_time  = time.time()
    latency   = (end_time - start_time) * 1000
    fps       = 1.0 / (end_time - prev_time + 1e-9)
    prev_time = end_time

    cv2.putText(frame, f"FPS: {fps:.1f}",             (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    cv2.putText(frame, f"Latency: {latency:.1f} ms",  (10, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    cv2.imshow('Emotion Recognition — YOLOv8n-face', frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('s'):
        # Simpan snapshot frame
        snap_path = os.path.join(
            log_dir,
            f"snapshot_{datetime.now().strftime('%H%M%S')}.jpg"
        )
        cv2.imwrite(snap_path, frame)
        print(f"[INFO] Snapshot disimpan: {snap_path}")

# ===============================
# RINGKASAN AKHIR SESI
# ===============================
csv_file.close()
cap.release()
cv2.destroyAllWindows()

elapsed = (datetime.now() - session_start).total_seconds()
print("\n" + "="*45)
print("       RINGKASAN SESI DETEKSI EMOSI")
print("="*45)
print(f"  Durasi sesi  : {int(elapsed//60):02d}:{int(elapsed%60):02d}")
print(f"  Log CSV      : {log_path}")
print("-"*45)
print(f"  {'Emosi':<10} {'Durasi':>8} {'%':>6} {'Conf Rata²':>12}")
print("-"*45)
dominant = max(emotion_durations, key=emotion_durations.get)
for e in EMOTIONS:
    dur   = emotion_durations[e]
    pct   = (dur / elapsed * 100) if elapsed > 0 else 0
    avg_c = (sum(conf_accum[e]) / len(conf_accum[e]) * 100) if conf_accum[e] else 0
    mark  = " ◀ DOMINAN" if e == dominant else ""
    print(f"  {e:<10} {dur:>7.1f}s {pct:>5.1f}% {avg_c:>10.1f}%{mark}")
print("="*45)
print("[INFO] Selesai.")

[INFO] Log disimpan ke: C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\emotion_log_20260625_012245.csv
Tekan 'q' untuk keluar  |  's' untuk simpan snapshot

       RINGKASAN SESI DETEKSI EMOSI
  Durasi sesi  : 00:54
  Log CSV      : C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\emotion_log_20260625_012245.csv
---------------------------------------------
  Emosi        Durasi      %   Conf Rata²
---------------------------------------------
  Angry         10.2s  18.6%       86.6%
  Drowsy         3.7s   6.7%       89.1%
  Netral        35.7s  65.1%       85.2% ◀ DOMINAN
  Sad            0.0s   0.0%        0.0%
[INFO] Selesai.


In [1]:
# ── REAL-TIME PREDICTION + ALERT + REPORT (YOLOv8n-face) ─────────────
import winsound
import cv2
import time
import csv
import os
from datetime import datetime
from ultralytics import YOLO
from collections import deque

# ===============================
# LOAD MODEL
# ===============================
emotion_model = YOLO(
    r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\best1.pt'
)
face_model = YOLO(
    r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\yolov8n-face.pt'
)

EMOTIONS = list(emotion_model.names.values())

COLORS = {
    'Angry'  : (0,   0,   255),
    'Drowsy' : (0,   128, 0  ),
    'Netral' : (0,   255, 0  ),
    'Sad'    : (128, 0,   128),
}

THRESHOLD = {
    "Angry"  : 0.60,
    "Drowsy" : 0.55,
    "Netral" : 0.60,
    "Sad"    : 0.30,
}

min_conf = THRESHOLD.get("Netral", 0.50)

# ===============================
# KAMERA
# ===============================
cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
for _ in range(5):
    cap.read()
if not cap.isOpened():
    raise RuntimeError("[ERROR] Kamera tidak bisa dibuka.")

# ===============================
# STATE
# ===============================
prev_time      = time.time()
emotion_buffer = deque(maxlen=3)

# ── REPORT STATE ──────────────────────────────────────────────────────
session_start     = datetime.now()
emotion_counts    = {e: 0   for e in EMOTIONS}
emotion_durations = {e: 0.0 for e in EMOTIONS}
conf_accum        = {e: []  for e in EMOTIONS}

last_stable       = None
last_stable_start = time.time()

report_interval   = 5.0
last_report_time  = time.time()
overlay_lines     = []

# ── METRIK PERFORMA ───────────────────────────────────────────────────
fps_history       = []        # semua nilai FPS per frame
latency_history   = []        # semua nilai latency (ms) per frame
total_frames      = 0         # total frame yang dibaca kamera
detected_frames   = 0         # frame yang berhasil mendeteksi wajah + emosi
skipped_frames    = 0         # frame yang dilewati (conf terlalu rendah)

# ── CSV LOG ───────────────────────────────────────────────────────────
log_dir  = r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika'
log_path = os.path.join(log_dir, f"emotion_log_{session_start.strftime('%Y%m%d_%H%M%S')}.csv")

csv_file   = open(log_path, 'w', newline='', encoding='utf-8')
csv_writer = csv.writer(csv_file)
csv_writer.writerow(['timestamp', 'emotion', 'confidence', 'fps', 'latency_ms'])

print(f"[INFO] Log disimpan ke: {log_path}")
print("Tekan 'q' untuk keluar  |  's' untuk simpan snapshot")

# ── HELPER: build overlay ─────────────────────────────────────────────
def build_overlay(emotion_durations, emotion_counts, conf_accum, session_start):
    lines   = []
    elapsed = (datetime.now() - session_start).total_seconds()
    lines.append(f"Sesi : {int(elapsed//60):02d}:{int(elapsed%60):02d}")
    lines.append("─────────────────────")
    dominant = max(emotion_durations, key=emotion_durations.get)
    for e in EMOTIONS:
        dur   = emotion_durations[e]
        pct   = (dur / elapsed * 100) if elapsed > 0 else 0
        avg_c = (sum(conf_accum[e]) / len(conf_accum[e]) * 100) if conf_accum[e] else 0
        marker = " ◀" if e == dominant else ""
        lines.append(f"{e:<8} {dur:5.1f}s {pct:4.0f}% c:{avg_c:3.0f}%{marker}")
    lines.append("─────────────────────")
    lines.append(f"Dominan: {dominant}")
    return lines

# ── HELPER: draw overlay ──────────────────────────────────────────────
def draw_report_overlay(frame, lines, x=10, y=75):
    font       = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.48
    thickness  = 1
    pad        = 6
    line_h     = 18

    max_w = max(cv2.getTextSize(l, font, font_scale, thickness)[0][0] for l in lines)
    box_h = len(lines) * line_h + pad * 2
    box_w = max_w + pad * 2

    overlay = frame.copy()
    cv2.rectangle(overlay, (x, y), (x + box_w, y + box_h), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

    for i, line in enumerate(lines):
        ty    = y + pad + (i + 1) * line_h - 3
        color = (200, 200, 200)
        for emo, col in COLORS.items():
            if line.strip().startswith(emo):
                color = col
                break
        if "Dominan" in line or "Sesi" in line or "─" in line:
            color = (0, 220, 220)
        cv2.putText(frame, line, (x + pad, ty), font, font_scale, color, thickness)

# ── HELPER: ringkasan akhir ───────────────────────────────────────────
def print_summary(session_start, log_path,
                  emotion_durations, conf_accum,
                  fps_history, latency_history,
                  total_frames, detected_frames, skipped_frames):

    elapsed = (datetime.now() - session_start).total_seconds()
    dominant = max(emotion_durations, key=emotion_durations.get)

    # Hitung metrik performa
    avg_fps      = sum(fps_history)     / len(fps_history)     if fps_history     else 0
    min_fps      = min(fps_history)                             if fps_history     else 0
    max_fps      = max(fps_history)                             if fps_history     else 0
    avg_lat      = sum(latency_history) / len(latency_history)  if latency_history else 0
    min_lat      = min(latency_history)                         if latency_history else 0
    max_lat      = max(latency_history)                         if latency_history else 0
    det_rate     = (detected_frames / total_frames * 100)       if total_frames    else 0

    # Rata-rata confidence semua emosi yang terdeteksi
    all_conf = [c for lst in conf_accum.values() for c in lst]
    avg_acc  = (sum(all_conf) / len(all_conf) * 100) if all_conf else 0

    W = 52  # lebar tabel

    def sep(char="─"): return char * W

    print("\n" + "═"*W)
    print("        RINGKASAN SESI DETEKSI EMOSI")
    print("═"*W)
    print(f"  Waktu mulai  : {session_start.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Durasi sesi  : {int(elapsed//60):02d}:{int(elapsed%60):02d}  ({elapsed:.1f} detik)")
    print(f"  Log CSV      : {os.path.basename(log_path)}")

    # ── Performa ──────────────────────────────────────────────────────
    print("\n" + sep())
    print("  PERFORMA SISTEM")
    print(sep())
    print(f"  {'Metrik':<22} {'Rata-rata':>10} {'Min':>8} {'Max':>8}")
    print(sep("·"))
    print(f"  {'FPS':<22} {avg_fps:>10.1f} {min_fps:>8.1f} {max_fps:>8.1f}")
    print(f"  {'Latency (ms)':<22} {avg_lat:>10.1f} {min_lat:>8.1f} {max_lat:>8.1f}")

    # ── Detection rate ────────────────────────────────────────────────
    print("\n" + sep())
    print("  DETECTION RATE")
    print(sep())
    print(f"  Total frame dibaca      : {total_frames:>6}")
    print(f"  Frame terdeteksi        : {detected_frames:>6}  ({det_rate:.1f}%)")
    print(f"  Frame dilewati (filter) : {skipped_frames:>6}  ({100-det_rate:.1f}%)")
    print(f"  Detection success rate  : {det_rate:>5.1f}%")

    # ── Accuracy (avg confidence) ─────────────────────────────────────
    print("\n" + sep())
    print("  AKURASI (RATA-RATA CONFIDENCE PER EMOSI)")
    print(sep())
    print(f"  {'Emosi':<10} {'Conf Rata²':>12} {'Sampel':>8}")
    print(sep("·"))
    for e in EMOTIONS:
        avg_c  = (sum(conf_accum[e]) / len(conf_accum[e]) * 100) if conf_accum[e] else 0
        n      = len(conf_accum[e])
        print(f"  {e:<10} {avg_c:>11.1f}% {n:>8}")
    print(sep("·"))
    print(f"  {'KESELURUHAN':<10} {avg_acc:>11.1f}%")

    # ── Distribusi emosi ──────────────────────────────────────────────
    print("\n" + sep())
    print("  DISTRIBUSI EMOSI")
    print(sep())
    print(f"  {'Emosi':<10} {'Durasi':>8} {'%':>7} {'Bar':<20}")
    print(sep("·"))
    for e in EMOTIONS:
        dur  = emotion_durations[e]
        pct  = (dur / elapsed * 100) if elapsed > 0 else 0
        bar  = "█" * int(pct / 5)   # 1 blok = 5%
        mark = " ◀ DOMINAN" if e == dominant else ""
        print(f"  {e:<10} {dur:>7.1f}s {pct:>6.1f}% {bar:<20}{mark}")

    print("\n" + "═"*W)
    print("[INFO] Selesai.")

# ═════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ═════════════════════════════════════════════════════════════════════
while True:
    start_time = time.time()

    ret, frame = cap.read()
    total_frames += 1

    if not ret or frame is None:
        continue

    frame = cv2.resize(frame, (640, 480))

    # ── Deteksi wajah ─────────────────────────────────────────────────
    face_results = face_model.predict(frame, conf=0.5, imgsz=640, verbose=False)
    boxes        = face_results[0].boxes
    stable_emotion = None
    frame_detected = False

    if boxes is not None and len(boxes) > 0:
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            H, W = frame.shape[:2]
            PAD  = 10
            x1c  = max(0, x1 - PAD);  y1c = max(0, y1 - PAD)
            x2c  = min(W, x2 + PAD);  y2c = min(H, y2 + PAD)

            face_crop = frame[y1c:y2c, x1c:x2c]
            if face_crop.size == 0:
                continue

            # Preprocess
            gray_crop  = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
            clahe      = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            gray_crop  = clahe.apply(gray_crop)
            gray_crop  = cv2.resize(gray_crop, (48, 48))
            face_input = cv2.cvtColor(gray_crop, cv2.COLOR_GRAY2BGR)

            # Prediksi emosi
            results   = emotion_model.predict(face_input, imgsz=64, verbose=False)
            probs     = results[0].probs
            top1      = probs.top1
            top1_conf = float(probs.top1conf)
            top2_conf = float(probs.top5conf[1]) if len(probs.top5conf) > 1 else 0.0
            emotion   = EMOTIONS[top1]
            conf_gap  = top1_conf - top2_conf

            # Filter
            if emotion == "Sad" and top1_conf < 0.60 and conf_gap < 0.20:
                skipped_frames += 1
                continue
            if emotion != "Sad" and top1_conf < min_conf:
                skipped_frames += 1
                continue

            # Buffer + voting
            emotion_buffer.append((emotion, top1_conf))
            score = {}
            for e, c in emotion_buffer:
                score[e] = score.get(e, 0) + c
            stable_emotion = max(score, key=score.get)

            if emotion == "Sad" and top1_conf > 0.30:
                stable_emotion = "Sad"
            if stable_emotion == "Sad" and score.get("Sad", 0) < 1.2:
                stable_emotion = "Netral"

            # Drowsy alert
            if stable_emotion == "Drowsy" and top1_conf > 0.70:
                winsound.Beep(1000, 300)

            # Gambar bbox
            color = COLORS.get(stable_emotion, (255, 255, 255))
            label = f"{stable_emotion} {top1_conf*100:.0f}%"
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.rectangle(frame, (x1, y1 - 30), (x2, y1), color, -1)
            cv2.putText(frame, label, (x1 + 5, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            # Statistik
            now = time.time()
            emotion_counts[stable_emotion] += 1
            conf_accum[stable_emotion].append(top1_conf)

            if stable_emotion != last_stable:
                last_stable       = stable_emotion
                last_stable_start = now
            else:
                emotion_durations[stable_emotion] += now - last_stable_start
                last_stable_start = now

            frame_detected = True

            # CSV log
            end_time_log = time.time()
            fps_log      = 1.0 / (end_time_log - prev_time + 1e-9)
            lat_log      = (end_time_log - start_time) * 1000
            csv_writer.writerow([
                datetime.now().strftime('%H:%M:%S.%f')[:-3],
                stable_emotion,
                f"{top1_conf:.4f}",
                f"{fps_log:.1f}",
                f"{lat_log:.1f}"
            ])

    if frame_detected:
        detected_frames += 1

    # ── Refresh overlay ───────────────────────────────────────────────
    now = time.time()
    if now - last_report_time >= report_interval:
        overlay_lines    = build_overlay(emotion_durations, emotion_counts,
                                         conf_accum, session_start)
        last_report_time = now

    if overlay_lines:
        draw_report_overlay(frame, overlay_lines)

    # ── FPS & Latency ─────────────────────────────────────────────────
    end_time  = time.time()
    latency   = (end_time - start_time) * 1000
    fps       = 1.0 / (end_time - prev_time + 1e-9)
    prev_time = end_time

    fps_history.append(fps)
    latency_history.append(latency)

    cv2.putText(frame, f"FPS: {fps:.1f}",            (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    cv2.putText(frame, f"Latency: {latency:.1f} ms", (10, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    cv2.imshow('Emotion Recognition — YOLOv8n-face', frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('s'):
        snap_path = os.path.join(
            log_dir, f"snapshot_{datetime.now().strftime('%H%M%S')}.jpg"
        )
        cv2.imwrite(snap_path, frame)
        print(f"[INFO] Snapshot disimpan: {snap_path}")

# ═════════════════════════════════════════════════════════════════════
# RINGKASAN AKHIR
# ═════════════════════════════════════════════════════════════════════
csv_file.close()
cap.release()
cv2.destroyAllWindows()

print_summary(
    session_start, log_path,
    emotion_durations, conf_accum,
    fps_history, latency_history,
    total_frames, detected_frames, skipped_frames
)

[INFO] Log disimpan ke: C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\emotion_log_20260626_211354.csv
Tekan 'q' untuk keluar  |  's' untuk simpan snapshot

════════════════════════════════════════════════════
        RINGKASAN SESI DETEKSI EMOSI
════════════════════════════════════════════════════
  Waktu mulai  : 2026-06-26 21:13:54
  Durasi sesi  : 00:15  (15.8 detik)
  Log CSV      : emotion_log_20260626_211354.csv

────────────────────────────────────────────────────
  PERFORMA SISTEM
────────────────────────────────────────────────────
  Metrik                  Rata-rata      Min      Max
····················································
  FPS                          10.7      1.3     14.7
  Latency (ms)                106.1     65.4    785.2

────────────────────────────────────────────────────
  DETECTION RATE
────────────────────────────────────────────────────
  Total frame dibaca      :    142
  Frame terdeteksi        :    131  (92.3%)
  Fra